# Human2API — Build real Android Companion APK

هذا الدفتر يبني **APK حقيقي** من GitHub باستخدام Google Colab (ليس ملف ZIP مُعاد تسميته).

**الخطوات:** افتح Runtime → Run all، وافق على ربط Google Drive، ثم راجع رسائل التراخيص إذا ظهرت. بعد النجاح ستجد التطبيق في `My Drive/Human2API-builds/`.

يبني إصدار **debug موقَّعًا بمفتاح اختبار** للاستخدام على جهازك. لا ترفعه إلى Google Play بوصفه إصدار إنتاج. التطبيق يستخدم Accessibility؛ لا تفعّل الخدمة إلا بعد مراجعة الأذونات وبموافقتك.

قد يستغرق تنزيل Android SDK وGradle وMaven بعض الوقت، ويحتاج Runtime متصلًا بالإنترنت. إذا فشل البناء سيُحفظ سجل الخطأ في Drive.


In [ ]:
import os, sys, shutil, hashlib, pathlib, subprocess, urllib.request, zipfile, datetime, textwrap
from google.colab import drive

drive.mount('/content/drive')
root = pathlib.Path('/content/human2api-apk-build')
root.mkdir(parents=True, exist_ok=True)
outdir = pathlib.Path('/content/drive/MyDrive/Human2API-builds')
outdir.mkdir(parents=True, exist_ok=True)
repo = root / 'repo'
sdk = root / 'android-sdk'
gradle_version = '8.7'

def run(args, *, cwd=None, env=None, timeout=1800, logfile=None):
    print('$', ' '.join(map(str,args)),flush=True)
    p = subprocess.run([str(x) for x in args], cwd=cwd, env=env,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                       text=True, errors='replace', timeout=timeout)
    print(p.stdout[-6000:],flush=True)
    if logfile:
        pathlib.Path(logfile).write_text(p.stdout, encoding='utf-8')
    if p.returncode:
        raise RuntimeError('Command failed with exit code %d: %s' % (p.returncode, ' '.join(map(str,args))))
    return p.stdout

def download(url, target):
    target=pathlib.Path(target)
    if target.exists() and target.stat().st_size>1000000:
        return target
    print('Downloading',url,flush=True)
    with urllib.request.urlopen(url,timeout=180) as response, target.open('wb') as output:
        shutil.copyfileobj(response,output)
    print('Downloaded',target.stat().st_size,'bytes',flush=True)
    return target

try:
    # Colab Linux JDK17 (Android Gradle Plugin 8.2.2 requires JDK17).
    if not pathlib.Path('/usr/lib/jvm/java-17-openjdk-amd64/bin/javac').exists():
        run(['bash','-lc','apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless'],timeout=1200)
    env = os.environ.copy()
    env['JAVA_HOME']='/usr/lib/jvm/java-17-openjdk-amd64'
    env['PATH']=env['JAVA_HOME']+'/bin:'+env.get('PATH','')
    env['ANDROID_HOME']=str(sdk)
    env['ANDROID_SDK_ROOT']=str(sdk)
    env['GRADLE_USER_HOME']=str(root / 'gradle-cache')
    run(['java','-version'],env=env)

    # Compile from the latest public GitHub source, not a previous ZIP snapshot.
    if not (repo / '.git').is_dir():
        run(['git','clone','--depth','1','--branch','main',
             'https://github.com/karemfree1282-gif/human2api.git',repo],timeout=300)
    else:
        run(['git','fetch','--depth','1','origin','main'],cwd=repo,timeout=300)
        run(['git','reset','--hard','origin/main'],cwd=repo)
    sha=run(['git','rev-parse','HEAD'],cwd=repo).strip()
    print('Source commit:',sha)

    # Official Android command-line tools and API level 34.
    sdkmanager = sdk / 'cmdline-tools/latest/bin/sdkmanager'
    if not sdkmanager.is_file():
        sdk.mkdir(parents=True,exist_ok=True)
        archive=download('https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip',root/'android-cmdline-tools.zip')
        dest=sdk/'cmdline-tools'
        dest.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            z.extractall(dest)
        (dest/'cmdline-tools').rename(dest/'latest')
    if not sdkmanager.is_file():
        raise RuntimeError('Android SDK command-line tool missing after extraction')
    sdkmanager.chmod(sdkmanager.stat().st_mode|0o111)
    # Running this cell explicitly accepts Android SDK licenses for this temporary Colab environment.
    run(['bash','-lc','yes | "$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager" --sdk_root="$ANDROID_HOME" --licenses'],env=env,timeout=900)
    run([sdkmanager,'--sdk_root='+str(sdk),'platforms;android-34','build-tools;34.0.0','platform-tools'],env=env,timeout=1500)
    if not (sdk/'platforms/android-34/android.jar').is_file():
        raise RuntimeError('SDK API 34 android.jar was not installed')

    # Official Gradle distribution with pinned SHA256.
    gradle = root / f'gradle-{gradle_version}/bin/gradle'
    if not gradle.is_file():
        archive=download('https://services.gradle.org/distributions/gradle-8.7-bin.zip',root/'gradle-8.7-bin.zip')
        h=hashlib.sha256(archive.read_bytes()).hexdigest()
        expected='544c35d6bd849ae8a5ed0bcea39ba677dc40f49df7d1835561582da2009b961d'
        if h!=expected:
            raise RuntimeError('Gradle archive SHA256 mismatch: '+h)
        with zipfile.ZipFile(archive) as z:
            z.extractall(root)
    gradle.chmod(gradle.stat().st_mode|0o111)

    logpath=outdir / ('Human2API-Companion-build-'+sha[:12]+'.log')
    run([gradle,'--no-daemon',':app:assembleDebug','--stacktrace'],cwd=repo/'android-companion',env=env,timeout=2400,logfile=logpath)
    apk=repo/'android-companion/app/build/outputs/apk/debug/app-debug.apk'
    if not apk.is_file():
        raise RuntimeError('Build finished without app-debug.apk')
    with zipfile.ZipFile(apk) as z:
        contents=set(z.namelist())
    if not {'AndroidManifest.xml','classes.dex','resources.arsc'}.issubset(contents):
        raise RuntimeError('APK is missing a compiled manifest, DEX or resource table')
    filename=f'Human2API-Companion-{sha[:12]}-debug.apk'
    saved=outdir/filename
    shutil.copy2(apk,saved)
    digest=hashlib.sha256(saved.read_bytes()).hexdigest()
    (outdir/(filename+'.sha256')).write_text(digest+'  '+filename+'\n',encoding='utf-8')
    print('\\nAPK BUILD SUCCEEDED')
    print('Git commit:',sha)
    print('APK in Google Drive:',saved)
    print('APK bytes:',saved.stat().st_size)
    print('SHA256:',digest)
    try:
        from google.colab import files
        files.download(str(saved))
    except Exception as exc:
        print('Direct browser download not available; APK is still saved in Drive:',exc)
except Exception as exc:
    error_log=outdir/'Human2API-Companion-last-build-error.txt'
    error_log.write_text(repr(exc)+'\n',encoding='utf-8')
    print('BUILD FAILED:',repr(exc))
    print('Failure report in Drive:',error_log)
    raise
